In [1]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append('..')


import utilities.functions as functions

from utilities.functions import (
    #load_data,
    check_key_uniqueness,
    merge_df,
    load_orders,
   orders_month_cli
)
from utilities.load_data import (

   load_data,
   count_null

)

In [2]:
from pathlib import Path
BASE_PATH = Path("/Users/maceli/ifood_cs/dados") 



Load the data --- se necessario salvar em stage - neste momento o estara comentado

In [3]:
URL_CONSUMER = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/consumer.csv.gz"
df_consumer = load_data(URL_CONSUMER)[["customer_id", "active","created_at"]]

In [4]:
URL_RESTAURANT ="https://data-architect-test-source.s3-sa-east-1.amazonaws.com/restaurant.csv.gz"
df_restaurant= load_data(URL_RESTAURANT)

In [5]:
ab_test_url = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/ab_test_ref.tar.gz"
df_ab = load_data(ab_test_url)

In [6]:
#save_parquet(df_consumer, "stage", "df_consumer.parquet")
#save_parquet(df_restaurant, "stage", "df_restaurant.parquet")
#save_parquet(df_consumer, "stage", "df_consumer.parquet")

Bronze layer - verify duplicates e nulo. e se necessario remover

In [7]:
check_key_uniqueness(df_consumer, ["customer_id","active","created_at"])

✅ Colunas ['customer_id', 'active', 'created_at'] são NOT NULL e UNIQUE.


(True, None, None, None)

In [8]:
check_key_uniqueness(df_restaurant, ["id"])

✅ Colunas ['id'] são NOT NULL e UNIQUE.


(True, None, None, None)

In [9]:
check_key_uniqueness(df_ab, ["customer_id","is_target"])
print(df_ab[df_ab["customer_id"].isna()])


❌ Colunas ['customer_id', 'is_target'] contêm valores nulos.

Soma de nulos por coluna:
customer_id    1
is_target      0
dtype: int64

Índices com nulos:
[81149]
      customer_id is_target
81149         NaN    target


Valida nullos

In [10]:
df_consumer_null=count_null(df_consumer)
df_consumer_null.head()

,Total,Percent
customer_id,0,0.0
active,0,0.0
created_at,0,0.0


In [11]:
df_restaurant_null=count_null(df_restaurant)
df_restaurant_null.head()

,Total,Percent
minimum_order_value,95,0.013028
delivery_time,1,0.000137
id,0,0.000000
created_at,0,0.000000
enabled,0,0.000000


In [12]:
df_ab_null=count_null(df_ab)
df_ab_null.head()

,Total,Percent
customer_id,1,0.000001
is_target,0,0.000000


In [13]:
df_ab_np = df_ab[~df_ab["customer_id"].isna()]

Antes de carregar a base ordens sera definido o publico todal, e da base de ordem serao filtrados somentes os clientes elegiceis

In [14]:
df_publico=merge_df(df_ab_np,df_consumer,['customer_id'],'left')

In [15]:
clientes_mes = (
    df_publico.groupby(['active', 'is_target'], dropna=False)
              ['customer_id']
              .nunique()
              .reset_index(name='numero_clientes_distintos')
)
clientes_mes

,active,is_target,numero_clientes_distintos
0,True,control,359700
1,True,target,444861
2,NaN,control,129
3,NaN,target,181
4,False,control,713
5,False,target,882


In [16]:
df_publico = df_publico.dropna(subset=['active'])

In [17]:
clientes_mes = (
    df_publico.groupby(['active', 'is_target'], dropna=False)
              ['customer_id']
              .nunique()
              .reset_index(name='numero_clientes_distintos')
)
clientes_mes

,active,is_target,numero_clientes_distintos
0,False,control,713
1,False,target,882
2,True,control,359700
3,True,target,444861


Publico definido e todos os clientes marcados no teste a/b e existentes na base de clientes

Da base de ordens serao filtrados todos os clientes com orden nos meses de de dezembro e janeiro

In [18]:
URL_ORDERS = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/order.json.gz"


COLUMNS_TO_DROP = [
    'cpf','customer_name','delivery_address_city','delivery_address_country',
    'delivery_address_district','delivery_address_external_id',
    'delivery_address_latitude','delivery_address_longitude',

    'delivery_address_zip_code','items',
    'merchant_latitude','merchant_longitude','merchant_timezone',
    'order_scheduled','order_scheduled_date'
]

customer_ids = df_ab["customer_id"].astype(str).unique()

df_orders = load_orders(
    url=URL_ORDERS,
    customer_ids=customer_ids,
    columns_to_drop=COLUMNS_TO_DROP
)

df_orders.head()


,customer_id,delivery_address_state,merchant_id,order_created_at,order_id,order_total_amount,origin_platform
0,7ba88a68bb2a3504c6bd37a707af57a0b8d6e110a551c7...,SP,a992a079a651e699d9149423761df2427c0e3af0a2a1b5...,2019-01-17T22:50:06.000Z,33e0612d62e5eb42aba15b58413137e441fbe906de2feb...,46.0,ANDROID
1,078acecdcf7fa89d356bfa349f14a8219db1ee161ce28a...,SP,5152f28ee0518b8803ccf0a4096eb2ff8b81e9491861c9...,2019-01-17T17:51:26.000Z,148c4353a2952f3fe7973547283265eb22b575fb712ed2...,104.5,ANDROID
2,0e38a3237b5946e8ab2367b4f1a3ae6e77f1e215bc760c...,SP,b6096419455c35d06105a5ef0d25c51f9dd40e1e99ac33...,2019-01-17T22:53:47.000Z,c37e495a91b498bb7b70a9e09ac115d0cdd443f152dc11...,35.0,IOS
3,cab1a004b7206d07910092c515a79834fea0a03d7d9054...,SP,082bfdcdf6ccdc343e3c4d25ee376b5b6ca7e96ad8b04e...,2019-01-17T23:56:53.000Z,b4df94142d21354611247da9ca94f870c09b93989b531a...,40.8,IOS
4,aa7edf5b166b8c843aec3b96dc561222888734f3879123...,ES,d7adb764bac29ccb77fb8f746ffbd531bf05ec30a7e130...,2019-01-17T23:40:53.000Z,4ff64b33b272c1886df21b63272220af6a82d1667dba70...,48.5,ANDROID


In [19]:
print(check_key_uniqueness(df_orders, ["customer_id","merchant_id","order_id"]))
print(check_key_uniqueness(df_orders, ["customer_id","merchant_id","order_id","order_created_at"]))

❌ Colunas ['customer_id', 'merchant_id', 'order_id'] possuem duplicações.
(False,                                                customer_id  \
1192509  c94d0872922e87c7f23669afcce3cc700d82ed75080a59...   
1192510  1e2af3429ee49b319a095258707980a7f1da2c2f1b6ca9...   
1192511  9a68122c178c32840eef9421530a375629d586ef9a7643...   
1192512  afdf6cab94f0e58e8a03940bb23243dc73263ad217205c...   
1192513  e9f79cc65b905e1e8cae0687605be386bc0275f6ec46ad...   
...                                                    ...   
3662316  648ae0e610811af0fccbe557b9a63a55c6e46adeeceb0b...   
3662317  5cab7f42316c5815d151d1fd0eebaecf9e6e53681257f0...   
3662318  1e91e110ba83f466ddbdb8ea448940e39e4e5ce16925e5...   
3662319  588becd71bc59b9a17ffcafe5823ce77777f308296f6f0...   
3662320  50862fb1670635160c98cd292768893dd65df05e5ae38e...   

        delivery_address_state  \
1192509                     BA   
1192510                     SP   
1192511                     SP   
1192512                     PR   
119

In [20]:
df_orders_null=count_null(df_orders)
df_orders_null.head()

,Total,Percent
origin_platform,2,5.461018e-07
customer_id,0,0.000000e+00
delivery_address_state,0,0.000000e+00
merchant_id,0,0.000000e+00
order_created_at,0,0.000000e+00


Add orders para a base de publico

In [21]:
df_publico_orders=merge_df(df_publico,df_orders,['customer_id'],'inner')

In [22]:
#df_publico_orders.to_parquet(BASE_PATH / "silver" / "df_publico_orders.parquet", index=False)

Construcao de chave unica, e sumarizacoes visao cliente

In [23]:
df_cliente,df_publico=orders_month_cli(df_publico_orders)

In [24]:
df_cliente.head()

,customer_id,is_target,order_created_month,num_pedidos_mes,num_pedidos_hist,total_amount_mes,ticket_medio
0,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,1,17,19,206.50,12.147059
1,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,12,2,19,22.50,11.250000
2,b821aa8372b8e5b82cdc283742757df8c45eecdd72adf4...,control,1,5,6,267.88,53.576000
3,b821aa8372b8e5b82cdc283742757df8c45eecdd72adf4...,control,12,1,6,20.00,20.000000
4,d425d6ee4c9d4e211b71da8fc60bf6c5336b2ea9af9cc0...,control,1,20,31,1300.89,65.044500


In [25]:
df_stats_hist = df_cliente.groupby(['num_pedidos_hist']).agg(
    total_clientes=('customer_id', 'nunique')  
).round(2)

df_stats_hist['pct_total'] = (df_stats_hist['total_clientes'] / df_stats_hist['total_clientes'].sum() * 100).round(2)

df_stats_hist.head(20)

,total_clientes,pct_total
num_pedidos_hist,,
1,182521,22.64
2,239366,29.69
3,70941,8.80
4,81498,10.11
5,38939,4.83
6,40129,4.98
7,23722,2.94
8,23231,2.88
9,15483,1.92


In [26]:
#df_publico_orders = pd.read_parquet(BASE_PATH / "silver" / "df_publico_orders.parquet")

Uma linha por cliente, com as variaveis necessarias

Salvar base visao cliente em gold layer

In [27]:
df_cliente.to_parquet(BASE_PATH / "gold" / "df_clientes.parquet", index=False)

In [28]:
df_publico.to_parquet(BASE_PATH / "gold" / "df_publico.parquet", index=False)